# Sufit: ile jest do wygrania na typologiach testowych

Krok 0 punktu „trening”. Dwa prerejestrowane eksperymenty z `trening.ipynb` (wczesne zatrzymanie, enkoder
resztkowy) pokazały, że metryka uczona z 9 typologii treningowych nie przenosi się przewidywalnie na nowe.
Zanim cokolwiek dalej, pytanie: **gdzie w ogóle jest sufit** na typologiach testowych każdego foldu i co go
wyznacza — etykiety czy cechy?

Dla każdego foldu, na jego 4 typologiach testowych, w jednostkach bramki (trafność zbalansowana):

1. **podłoga** — losowy, nietrenowany MLP (z `split.json` kampanii);
2. **5-shot na surowych cechach** — prototypy bez żadnego enkodera, te same 1000 epizodów co w bramce;
3. **ProtoNet 5-shot** — trening z pipeline'u (`fsl.training.loop.train`), ten sam co w kampanii;
4. **krzywa etykiet** — prototypy na surowych cechach przy 1, 2, 5, 10, 20 i wszystkich etykietach na typologię,
   pula/zbiór kontrolny rozłączne po grupach kont (60/40, jak w symulacji active learningu), 20 powtórzeń;
5. **sufit nadzorowany** — regresja logistyczna i las losowy z pełnymi etykietami, ta sama pula i zbiór kontrolny.

Headroom = sufit nadzorowany − prototypy przy 5 etykietach (pkt 4 przy m=5). Kryterium zapisane przed uruchomieniem:

- headroom ≥ 5 pp średnio po foldach → różnicę odbierają **etykiety**, nie trening enkodera; to robi pętla z analitykiem;
- headroom < 3 pp → typologie nie są rozdzielne w tych cechach; dźwignią są **cechy** (budowa przypadku), nie model;
- między 3 a 5 pp → wynik niejednoznaczny, decyzja po krzywej etykiet (gdzie się nasyca).

Dodatkowo: jeśli prototypy ze **wszystkimi** etykietami są ≥ 3 pp poniżej sufitu nadzorowanego, sama rodzina
„wzorzec = średnia” zostawia pieniądze na stole (granice typologii nie są kuliste) — wtedy inny klasyfikator na
tych samych etykietach jest kandydatem, bez uczenia metryki.

Nic się tu nie trenuje poza punktem 3 (ok. minuta na fold na CPU; `LICZ_PROTONET = False` go pomija).
Wyniki: tabela w notebooku, `wyniki/sufit.csv`, wykresy `wyniki/sufit_f<k>.png`.


In [ ]:
# --- KONFIGURACJA --------------------------------------------------------------
PROJECT, REGION, BUCKET = "dark-data-discovery", "us-central1", "dark-data-discovery-fsl-data"
DATASET = "samld-case"
SPLIT_URI = ""                 # split.json z kampanii; puste = najnowszy w buckecie; brak = foldy i podlogi z freeze'u
FOLDY = None                   # None = wszystkie foldy ze splitu; albo np. [3, 4]
LICZ_PROTONET = True           # trening z pipeline'u na kazdym foldzie (ok. 1 min/fold na CPU)
POWTORZENIA = 20               # losowania pula/kontrola dla krzywej etykiet i sufitu
SEED = 0


In [ ]:
# --- DANE (nie zmieniaj) ---------------------------------------------------------
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from fsl.config import TrainConfig
from fsl.data.manifest import fetch_manifest, get_geometry
from fsl.data.samld import build_task_samplers, load_frozen_samld
from fsl.training.loop import evaluate, set_seed, train

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/pipeline_config_samld.json").exists())
CFG = json.loads((ROOT / "configs/pipeline_config_samld.json").read_text())
BASE = CFG["training"]
WYNIKI_DIR = Path("wyniki")
WYNIKI_DIR.mkdir(exist_ok=True)
device = torch.device("cpu")                            # jak kontener treningowy; deterministycznie

ma_split = "split_uri" in TrainConfig.__dataclass_fields__
if ma_split and not SPLIT_URI:
    from google.cloud import storage
    docs = [b for b in storage.Client(project=PROJECT).list_blobs(BUCKET, prefix=f"splits/{DATASET}/")
            if b.name.endswith("/split.json")]
    if docs:
        SPLIT_URI = f"gs://{BUCKET}/{max(docs, key=lambda b: b.updated).name}"
if SPLIT_URI and not ma_split:
    raise SystemExit("ta wersja pakietu fsl nie zna split_uri — zaktualizuj repo (v1.1+) albo ustaw SPLIT_URI = ''")

manifest = fetch_manifest(PROJECT, BUCKET, DATASET)
in_features, _ = get_geometry(manifest, DATASET)


def konfig(fold):
    return TrainConfig(project=PROJECT, region=REGION, bucket=BUCKET, dataset=DATASET, fold=fold,
                       encoder="mlp", in_channels=in_features, image_size=1, seed=SEED,
                       n_way=BASE["n_way"], k_shot=BASE["k_shot"], query=BASE["query"],
                       train_iters=BASE["train_iters"], eval_every=BASE["eval_every"],
                       val_episodes=BASE["val_episodes"], test_episodes=BASE["test_episodes"],
                       lr=BASE["lr"], embedding_hid=BASE["embedding_hid"],
                       mlp_depth=BASE["mlp_depth"], mlp_dropout=BASE["mlp_dropout"], log_to_vertex=False,
                       **({"split_uri": SPLIT_URI} if ma_split else {}))


bundle, data_sha = load_frozen_samld(konfig(0))
schema, folds, stats = bundle["schema"], bundle["folds"], bundle["stats"]
W = bundle["windows"]
if SPLIT_URI:
    from fsl.data.split import load_split
    split_doc = load_split(SPLIT_URI, PROJECT)
    FLOORS, THRESHOLDS, SPLIT_SHA = split_doc["floors"], split_doc["thresholds"], split_doc["split_sha"]
else:
    legacy = Path.home() / "data_cache/samld_case_floors.json"
    assert legacy.exists(), "brak split.json i brak ~/data_cache/samld_case_floors.json"
    fl = json.loads(legacy.read_text())
    FLOORS, THRESHOLDS, SPLIT_SHA = fl["floors"], fl["thresholds"], ""
FOLDY = list(range(len(folds))) if FOLDY is None else list(FOLDY)

feats = schema["feature_names"]
heavy = [feats.index(c) for c in schema["log1p_features"]]
labels_all = W[schema["label_col"]].to_numpy().astype(str)
groups_all = W[schema["group_col"]].to_numpy()


def cechy_folda(fold):
    """Standaryzacja folda jak w build_task_samplers: log1p na ciezkich cechach, mu/sigma z typologii treningowych."""
    st = stats[str(fold)]
    X = W[feats].astype("float64").to_numpy().copy()
    X[:, heavy] = np.log1p(X[:, heavy])
    return ((X - np.asarray(st["mu"])) / np.asarray(st["sigma"])).astype("float32")


print(f"{DATASET} | split {SPLIT_SHA[:12] or 'freeze'} | dane {data_sha[:12]} | {len(W)} przypadkow | {in_features} cech")
for k in FOLDY:
    print(f"fold {k}: test {', '.join(folds[k]['test'])} | podloga {FLOORS[str(k)]:.4f} | prog {THRESHOLDS[str(k)]:.4f}")


In [ ]:
# --- METODY (nie zmieniaj) -------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score


class Identity(nn.Module):
    """Brak enkodera: prototypy liczone wprost na standaryzowanych cechach."""
    def forward(self, x):
        return x.flatten(1)


def epizodowo(model, cfg, n):
    """Trafnosc na n epizodach testowych folda, dokladnie protokolem bramki (fast_adapt, ten sam seed)."""
    _, _, test_tasks = build_task_samplers(bundle, cfg)
    return evaluate(model, test_tasks, cfg, n, device)


def centroidy(X, y, klasy):
    return np.stack([X[y == c].mean(0) for c in klasy])


def najblizszy_centroid(X, C, klasy):
    d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)
    return np.asarray(klasy)[d.argmin(1)]


def podzial_pula_kontrola(y, g, rng, udzial_puli=0.6):
    """Grupy kont rozlaczne miedzy pula (etykiety do wziecia) i kontrola (nigdy etykietowana) — jak w symulacji AL."""
    grupy = np.unique(g)
    rng.shuffle(grupy)
    n_pool = int(round(udzial_puli * len(grupy)))
    pool_g = set(grupy[:n_pool].tolist())
    pool = np.array([i for i in range(len(y)) if g[i] in pool_g])
    ev = np.array([i for i in range(len(y)) if g[i] not in pool_g])
    return pool, ev


def krzywa_etykiet(X, y, g, klasy, rng, budzety=(1, 2, 5, 10, 20, "all")):
    """Trafnosc zbalansowana najblizszego centroidu przy m etykietach na typologie (losowanych z puli)."""
    pool, ev = podzial_pula_kontrola(y, g, rng)
    out = {}
    for m in budzety:
        idx = []
        for c in klasy:
            kand = pool[y[pool] == c]
            if m == "all" or len(kand) <= m:
                idx.extend(kand.tolist())
            else:
                idx.extend(rng.choice(kand, m, replace=False).tolist())
        idx = np.array(idx)
        C = centroidy(X[idx], y[idx], klasy)
        out[str(m)] = balanced_accuracy_score(y[ev], najblizszy_centroid(X[ev], C, klasy))
    return out, pool, ev


def sufit_nadzorowany(X, y, pool, ev, seed):
    lr = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(X[pool], y[pool])
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=seed,
                                n_jobs=-1).fit(X[pool], y[pool])
    return {"logistic": balanced_accuracy_score(y[ev], lr.predict(X[ev])),
            "rf": balanced_accuracy_score(y[ev], rf.predict(X[ev])),
            "rf_per_class": {c: float(((rf.predict(X[ev]) == c) & (y[ev] == c)).sum() / max(1, (y[ev] == c).sum()))
                             for c in np.unique(y[ev])}}


In [ ]:
# --- POMIAR (nie zmieniaj) -------------------------------------------------------
wiersze, per_class = [], {}
for k in FOLDY:
    t0 = time.time()
    cfg = konfig(k)
    klasy = list(folds[k]["test"])
    X = cechy_folda(k)
    maska = np.isin(labels_all, klasy)
    Xk, yk, gk = X[maska], labels_all[maska], groups_all[maska]

    raw5 = epizodowo(Identity(), cfg, BASE["test_episodes"])
    proto = train(cfg)["test_accuracy_mean"] if LICZ_PROTONET else float("nan")

    krzywe, sufity = [], []
    for rep in range(POWTORZENIA):
        rng = np.random.default_rng(1000 * k + rep)
        kr, pool, ev = krzywa_etykiet(Xk, yk, gk, klasy, rng)
        krzywe.append(kr)
        sufity.append(sufit_nadzorowany(Xk, yk, pool, ev, seed=1000 * k + rep))
    kr_mean = {m: float(np.mean([kr[m] for kr in krzywe])) for m in krzywe[0]}
    kr_sd = {m: float(np.std([kr[m] for kr in krzywe], ddof=1)) for m in krzywe[0]}
    sl = float(np.mean([s["logistic"] for s in sufity]))
    srf = float(np.mean([s["rf"] for s in sufity]))
    per_class[k] = {c: float(np.mean([s["rf_per_class"].get(c, np.nan) for s in sufity])) for c in klasy}
    sufit = max(sl, srf)

    wiersze.append({"fold": k, "typologie": ", ".join(klasy), "n_przypadkow": int(maska.sum()),
                    "podloga": FLOORS[str(k)], "raw_5shot_epizody": float(raw5.mean()), "protonet_epizody": proto,
                    **{f"centroid_m{m}": kr_mean[m] for m in kr_mean},
                    "logistic_all": sl, "rf_all": srf, "sufit": sufit,
                    "headroom": sufit - kr_mean["5"], "luka_centroid_vs_sufit": sufit - kr_mean["all"]})
    print(f"fold {k} ({time.time() - t0:.0f} s): podloga {FLOORS[str(k)]:.3f} | raw 5-shot (epizody) {raw5.mean():.3f} | "
          f"ProtoNet {proto:.3f} | centroid m=5 {kr_mean['5']:.3f} ±{kr_sd['5']:.3f} | m=all {kr_mean['all']:.3f} | "
          f"logistic {sl:.3f} | RF {srf:.3f} | headroom {sufit - kr_mean['5']:+.3f}")

tab = pd.DataFrame(wiersze).set_index("fold")
tab.to_csv(WYNIKI_DIR / "sufit.csv")
pd.set_option("display.width", 200)
tab.round(3)


In [ ]:
# --- WERDYKT (nie zmieniaj) ------------------------------------------------------
h = tab["headroom"]
luka = tab["luka_centroid_vs_sufit"]
print(f"headroom (sufit - centroid m=5): srednio {h.mean():+.3f}, po foldach " +
      ", ".join(f"f{k} {v:+.3f}" for k, v in h.items()))
print(f"luka centroid(all) vs sufit: srednio {luka.mean():+.3f}, po foldach " +
      ", ".join(f"f{k} {v:+.3f}" for k, v in luka.items()))
if h.mean() >= 0.05:
    print("WERDYKT: headroom >= 5 pp — roznice odbieraja ETYKIETY (petla z analitykiem), nie trening enkodera.")
elif h.mean() < 0.03:
    print("WERDYKT: headroom < 3 pp — typologie nie sa rozdzielne w tych cechach; dzwignia to CECHY, nie model.")
else:
    print("WERDYKT: 3-5 pp — niejednoznaczne; patrz, gdzie krzywa etykiet sie nasyca.")
if luka.mean() >= 0.03:
    print("DODATKOWO: prototypy ze wszystkimi etykietami >= 3 pp ponizej sufitu — granice typologii nie sa kuliste; "
          "klasyfikator na tych samych etykietach (bez uczenia metryki) jest kandydatem.")
print("\nnajtrudniejsze typologie (recall RF przy pelnych etykietach):")
for k in FOLDY:
    najgorsza = min(per_class[k], key=per_class[k].get)
    print(f"  fold {k}: " + ", ".join(f"{c} {v:.2f}" for c, v in sorted(per_class[k].items(), key=lambda kv: kv[1]))
          + f"  -> najtrudniejsza: {najgorsza}")


In [ ]:
# --- WYKRESY (nie zmieniaj) ------------------------------------------------------
import matplotlib
import matplotlib.pyplot as plt

budzety = [c for c in tab.columns if c.startswith("centroid_m")]
xs = [c[len("centroid_m"):] for c in budzety]
for k in FOLDY:
    r = tab.loc[k]
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    ax.plot(range(len(xs)), [r[c] for c in budzety], marker="o", color="#1f4e8c", label="prototypy (surowe cechy)")
    ax.axhline(r["sufit"], color="#1a7f37", ls="--", label=f"sufit nadzorowany {r['sufit']:.3f}")
    ax.axhline(r["podloga"], color="#999", ls=":", label=f"podloga {r['podloga']:.3f}")
    if LICZ_PROTONET:
        ax.axhline(r["protonet_epizody"], color="#c62828", ls="-.", label=f"ProtoNet 5-shot {r['protonet_epizody']:.3f}")
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([("wszystkie" if x == "all" else x) for x in xs])
    ax.set_xlabel("etykiety na typologię")
    ax.set_ylabel("trafność zbalansowana")
    ax.set_title(f"fold {k}: {r['typologie']}", fontsize=10)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(WYNIKI_DIR / f"sufit_f{k}.png", dpi=150)
    plt.show()
